# Formula 1 Race Strategy Simulator
## Notebook 05: What-If Counterfactual Scenario Analysis

This notebook demonstrates **Phase 6: Scenario Analysis & Counterfactual What-If Engine**:
1. **Perturbation Manifolds**: Formalizing counterfactual changes to circuit, pit crew, and tyre parameters $\theta' = \mathcal{P}(\theta_0, \mathbf{p})$.
2. **Strategic Regret Theory**: Measuring the time cost of running the nominal plan under perturbed reality:
   $$\mathcal{R}(S; \theta') = T(S; \theta') - \min_{S'} T(S'; \theta')$$
3. **Core Research Scenarios from project.md**:
   * Pit loss variations ($\pm 3.0\text{s}$, botched stop $+10\text{s}$)
   * Extreme thermal degradation spikes ($+50\%$)
   * Soft tyre pace differential shifts
   * Sprint vs Full Grand Prix distance
   * Safety Car neutralization at Lap 18
   * 2026 Technical Regulation shifts
4. **Winning Strategy Pivots**: Identifying when tactical pivots become mandatory.

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.model import RaceModel
from src.config import BAHRAIN_CONFIG
from src.scenarios import ScenarioEngine, Scenario, PRESET_SCENARIOS

sns.set_theme(style="darkgrid")
print(f"Loaded {len(PRESET_SCENARIOS)} preset scenarios.")

### 1. The Scenario Analysis Engine

The `ScenarioEngine` perturbs physical model parameters while holding strategy candidate sets constant, allowing us to compute:
* Nominal optimal strategy time under counterfactual conditions
* Adapted optimal strategy time
* **Strategic Regret** $\mathcal{R}$: Time forfeited by not adapting the strategy
* **Strategy Pivot**: Whether the optimal tactic switches (e.g. 1-Stop $\to$ 2-Stop)

In [ ]:
model = RaceModel(BAHRAIN_CONFIG)
engine = ScenarioEngine(model)

print(f"Base Circuit: {model.circuit.name} ({model.circuit.total_laps} laps)")
print("Available Scenarios:")
for s_id, s_obj in PRESET_SCENARIOS.items():
    print(f"  * {s_id:20s}: {s_obj.name}")

### 2. Evaluating the Full What-If Scenario Matrix

Let us run all 11 pre-configured scenarios and compile a strategic impact table.

In [ ]:
results = engine.run_all_scenarios()

summary_rows = []
for res in results:
    summary_rows.append({
        "Scenario": res.scenario_name,
        "Nominal Winner": res.nominal_best_strategy,
        "Adapted Winner": res.adapted_best_strategy,
        "Strategy Pivot?": "YES" if res.strategy_pivoted else "No",
        "Regret (s)": round(res.strategic_regret, 2),
        "Time Delta (s)": round(res.delta_to_nominal_seconds, 2),
    })

df_scenarios = pd.DataFrame(summary_rows)
print(df_scenarios.to_string(index=False))

### 3. Visualizing Strategic Regret Across Scenarios

Strategic Regret $\mathcal{R}$ quantifies the competitive disaster of remaining on the baseline plan when track conditions mutate.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

names = [r.scenario_name for r in results]
regrets = [r.strategic_regret for r in results]
pivots = [r.strategy_pivoted for r in results]
colors = ["crimson" if p else "steelblue" for p in pivots]

bars = ax.barh(names, regrets, color=colors, edgecolor="black", height=0.6)
ax.set_title("Strategic Regret of Nominal Plan Under Counterfactual Scenarios", fontsize=13, fontweight="bold")
ax.set_xlabel("Strategic Regret (seconds forfeited)", fontsize=11)

# Annotate pivot flag
for bar, p in zip(bars, pivots):
    if p:
        ax.text(bar.get_width() + 0.3, bar.get_y() + 0.15, "PIVOT", color="crimson", fontweight="bold", fontsize=9)

plt.tight_layout()
plt.show()

### 4. Deep-Dive: Scenario 1 - Severe Tyre Degradation Spike (+50%)

When track temperature surges and degradation rises by $+50\%$, how does the race time and optimal strategy shift?

In [ ]:
res_deg = engine.evaluate_scenario("high_deg")

print(f"=== SCENARIO: {res_deg.scenario_name} ===")
print(f"Insight: {res_deg.summary_insight}")
print(f"Nominal Winner: {res_deg.nominal_best_strategy}")
print(f"Adapted Winner: {res_deg.adapted_best_strategy}")
print(f"Strategic Regret: {res_deg.strategic_regret:.2f} s")

print("
Strategy Outcomes under High Degradation:")
for outcome in res_deg.strategy_outcomes:
    print(f"  * {outcome.strategy_name:25s} Duration: {outcome.total_time:.2f}s  Delta to P1: {outcome.delta_to_p1:+.2f}s")

### 5. Deep-Dive: Scenario 2 - Safety Car Neutralization at Lap 18

Under a Safety Car, pit loss drops by $\approx 35\%$ because field speed is capped under the delta:
$$T_{pit}^{SC} = T_{transit} \cdot \frac{v_{racing}}{v_{delta}} + T_{stationary}$$

Does this trigger an immediate opportunistic pit stop?

In [ ]:
res_sc = engine.evaluate_scenario("safety_car_lap18")

print(f"=== SCENARIO: {res_sc.scenario_name} ===")
print(f"Insight: {res_sc.summary_insight}")
print(f"Strategy Pivot Occurred: {res_sc.strategy_pivoted}")
print(f"Strategic Regret: {res_sc.strategic_regret:.2f} s")

### 6. Deep-Dive: Scenario 3 - 2026 Technical Regulations Shift

The 2026 regulations introduce:
* Reduced fuel capacity ($70\text{ kg}$ vs $105\text{ kg}$)
* Active aerodynamics (reduced drag on straights)
* Altered power unit deployment ($50\%$ ICE / $50\%$ Electrical)

In [ ]:
res_2026 = engine.evaluate_scenario("regulations_2026")

print(f"=== SCENARIO: {res_2026.scenario_name} ===")
print(f"Insight: {res_2026.summary_insight}")
print(f"Nominal 2024 Winner: {res_2026.nominal_best_strategy}")
print(f"Adapted 2026 Winner: {res_2026.adapted_best_strategy}")
print(f"Time Delta to 2024 Baseline: {res_2026.delta_to_nominal_seconds:+.2f} s")